# Lab 02: Engression Neural Generator & Physics-Aware Loss Engine

Welcome to **Lab 02** of the **Tempest-ARKNET** Interactive Documentation Suite.

Here you will instantiate the real PyTorch **`Generator`** architecture from `tempest/generator/model.py` and run a fast synthetic training experiment (< 30 seconds on free Colab CPU) comparing standard `afcrps` against Tempest's full physics-aware loss stack (`afcrps` + `energy_score` + `shear_tail_loss` + `low_wind_loss`).

**How to use this lab:** Before each main code cell, pause on the **Predict → Run → Observe** callout and write down (or mentally commit to) your prediction before clicking **Run**.

**What you'll explore:**
1. **The Engression `Generator` Architecture** — AdaLN/FiLM noise blocks + linear slope bypass for climate extrapolation
2. **Proper Scoring Rules** — Almost-Fair CRPS (`afcrps`) and Unbiased Energy Score (`energy_score`)
3. **Physics-Aware Tail Losses** — Why standard CRPS smooths away rocket-critical wind shear, and how `shear_tail_loss` + `low_wind_loss` (`train.py:309-338`) fix it

### Before Running Setup: Why This Cell Checks the Runtime
This lab runs live PyTorch modules from `tempest.generator.model` (`Generator`, `afcrps`, `energy_score`). On Google Colab, `torch` is pre-installed and the compact experiments below run in under 30 seconds on a free CPU runtime—no GPU required.

In [ ]:
# @title 0. Environment Setup (Run Once)
MOUNT_DRIVE = False # @param {type:"boolean"}
REPO_URL = "https://github.com/senai-sorsa/Tempest-ARKNET.git" # @param {type:"string"}

import os, sys, subprocess, pathlib

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped (not running in Colab): {exc}")

repo_dir = pathlib.Path("/content/Tempest-ARKNET")
if not repo_dir.exists():
    # Allow running inside a local checkout if already inside the repo
    cwd = pathlib.Path.cwd()
    if (cwd / "tempest").is_dir() and (cwd / "tempest_weather_v2").is_dir():
        repo_dir = cwd
    else:
        print(f"Cloning {REPO_URL} into {repo_dir} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# Install lightweight dependencies quietly (torch is pre-installed on Google Colab)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['numpy', 'scipy', 'matplotlib', 'torch']], check=False)

print(f"Python: {sys.version.split()[0]} | Repo ready at: {repo_dir}")
if sys.version_info < (3, 12):
    print("Note: Full tempest_weather_v2.launch requires Python >= 3.12 (PEP 701 f-strings at launch.py:684).")
    print("      This lab imports Python 3.10/3.11-compatible submodules directly and runs cleanly on Colab CPU.")

## 1. The Engression `Generator` Architecture (AdaLN/FiLM + Linear Slope Bypass)

### Conceptual Intuition (Before the Math)
How does a neural network generate *hundreds of realistic, distinct weather days* for the exact same location, date, and time—without collapsing to a blurry average?

Think of Tempest's **`Generator`** (`tempest/generator/model.py`) like a **film studio**:
1. **The Set (`cond` → `keep_cols`):** Location, elevation, season (`sin/cos` day-of-year), and solar hour (`sin/cos`) establish the baseline climatology.
2. **Why AdaLN / FiLM Noise Conditioning in *Every* Block?** If you only concatenate random noise $z_\varepsilon$ at the very first input layer of a deep MLP, the network can easily learn small weights on the noise columns and collapse toward a deterministic mean. Instead, Tempest embeds a single low-dimensional noise vector $z_\varepsilon \in \mathbb{R}^{d}$ (via `self.embed` into a 128-dim vector) and uses **Adaptive LayerNorm / FiLM** (`LN(h) * (1 + s(z)) + b(z)`) inside *every* residual block. Every layer sees the exact same random draw, turning one baseline set into a calm morning, a gusty afternoon, or a strong jet-stream day—and because `self.film` is **zero-initialized**, each block starts training as a clean, stable LayerNorm identity!
3. **Why the Linear Slope Bypass (`linear_cols`)?** Slow climate drivers (`year`, `oni`, `wwv`, `qbo30`, `qbo50`) are deliberately **removed** from the deep nonlinear trunk (`keep_cols`). Why? Inside a deep MLP, four monthly climate indices act like a unique barcode that lets the network *memorize* individual training months (hurting held-out generalization!). Routing them through a zero-initialized **linear slope bypass** forces each driver to act as a straight-line trend whose slope depends smoothly on place, season, hour, and height—preventing month memorization and ensuring clean extrapolation to future launch years.

### Equation Bridge (`tempest/generator/model.py`)
$$h_{\ell+1} = h_\ell + \mathrm{FF}\!\big(\mathrm{LN}(h_\ell)\odot(1 + s_\ell(z_\varepsilon)) + b_\ell(z_\varepsilon)\big), \qquad \hat{y} = \mathrm{Head}(h_L) + \sum_{k \in \text{lin}} x_k\,\mathrm{Slope}_k(c_{\text{keep}})$$

> **Predict → Run → Observe (Generator Architecture & Tensor Shapes)**
> - **Predict:** Before running the next cell: (1) Out of the `13` conditioning columns in `BASE_FEATURES`, `5` climate drivers (`year, oni, wwv, qbo30, qbo50`) are routed to `linear_cols`—how many columns enter the nonlinear trunk `keep_cols`? (2) If we pass a batch of `B = 4` weather conditions and request `m = 8` ensemble members per condition for `4` profile variables × `44` heights + `3` surface scalars, what exact 3D shape will `samples_demo` have?
> - **Run:** Execute the next cell to instantiate the compact `Generator` and run a live forward pass.
> - **What to observe:** Verify the printed `linear_cols` indices, confirm the compact parameter count (~50k–200k vs. ~2.8M for the shipped `width=512, depth=4` checkpoint), and confirm `samples_demo.shape == (4, 8, 179)`.

In [ ]:
# @title 1. Instantiate Tempest's Real Engression Generator
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from tempest.generator.model import Generator, afcrps, energy_score
from tempest.generator.features import BASE_FEATURES
from tempest.profiles.to_heights import HEIGHTS, PROFILE
from tempest.profiles.diff import to_diff, from_diff

torch.manual_seed(42)
np.random.seed(42)

# Shipped config uses width=512, depth=4 (~2.8M params).
# Here we use a compact width=96, depth=2 instance for instant CPU experimentation.
n_cond = len(BASE_FEATURES)  # 13 conditioning features
n_out = len(PROFILE) * len(HEIGHTS) + 3  # 4 * 44 + 3 = 179
linear_cols = [BASE_FEATURES.index(k) for k in ("year", "oni", "wwv", "qbo30", "qbo50")]

gen = Generator(
    n_cond=n_cond,
    n_out=n_out,
    noise_dim=16,
    width=96,
    depth=2,
    linear_cols=linear_cols,
)

n_params = sum(p.numel() for p in gen.parameters())
cond_demo = torch.randn(4, n_cond)
samples_demo = gen(cond_demo, m=8)

print(f"BASE_FEATURES ({n_cond}): {BASE_FEATURES}")
print(f"Linear bypass indices ({len(linear_cols)}): {linear_cols}")
print(f"Compact Generator parameters: {n_params:,}")
print(f"Forward pass: cond {tuple(cond_demo.shape)} + m=8 -> samples {tuple(samples_demo.shape)}")

## 2. Proper Scoring Rules: Almost-Fair CRPS (`afcrps`) & Energy Score

### Conceptual Intuition (Before the Math)
Suppose you are judging a weather forecaster who gives you a small ensemble of $M = 4$ or $8$ possible wind profiles. How do you score them so they neither guess too narrowly (overconfident) nor too wildly (uninformative)?

Think of **Almost-Fair CRPS (`afcrps`)** as a **two-part dartboard wager**:
- **Accuracy (fit) term:** $\frac{1}{M}\sum_{i=1}^M |s_i - y|$ penalizes how far your darts land from the real weather outcome $y$.
- **Diversity (spread) bonus:** $-\frac{1 - \varepsilon}{2 M (M - 1)}\sum_{i=1}^M \sum_{j=1}^M |s_i - s_j|$ rewards your darts for spreading out to match nature's real uncertainty.

**Why the unbiased $\frac{1}{M(M-1)}$ spread denominator (instead of $\frac{1}{M^2}$)?**
Look closely at the double sum $\sum_{i,j} |s_i - s_j|$. Out of $M^2$ total pairs, the $M$ diagonal self-pairs ($i = j$) are always identically zero ($|s_i - s_i| = 0$), leaving only $M(M-1)$ **distinct** pairs. If you divided by $M^2$, the spread bonus would be shrunk by a factor of $(M-1)/M$ (a 25% underestimate when $M=4$!), tricking the generator into over-inflating its spread to compensate. Dividing by $M(M-1)$ makes the sample spread estimator **mathematically unbiased** for any finite ensemble size $M \ge 2$.

**Why $\alpha = 0.95$ instead of $\alpha = 1.0$ (pure Fair CRPS)?**
In `afcrps`, $\varepsilon = (1 - \alpha)/M$. When $\alpha = 1.0$ ($\varepsilon = 0$), the score is perfectly fair—but during stochastic gradient training with small $M$, pure Fair CRPS has a subtle degenerate optimum where one outlier member can drift far away while still collecting spread credit. Setting $\alpha = 0.95$ (Tempest's default, following ECMWF AIFS-CRPS) adds a gentle 5% anchor that eliminates runaway outliers while keeping spread nearly unbiased.

**Why divide `energy_score` by $\sqrt{D}$?**
`energy_score` generalizes the exact same unbiased $M(M-1)$ idea from 1D coordinates to the full $D$-dimensional profile vector using Euclidean norm $\|s_i - y\|_2$, rewarding realistic cross-height and cross-variable correlations. Because Euclidean norm in $\mathbb{R}^D$ grows like $\sqrt{D}$ for unit-variance coordinates, dividing by $\sqrt{D}$ keeps `energy_score` on the same $O(1)$ scale as `afcrps`.

> **Predict → Run → Observe (Proper Scoring Rules & Spread Optimality)**
> - **Predict:** Before running the next cell (where synthetic truth $y \sim \mathcal{N}(0, I)$ has true spread `1.00`): (1) At roughly what **Ensemble Spread Multiplier** will both `afCRPS` and `Energy Score / √D` hit their minimum? (2) If you lower `CRPS_ALPHA` from `0.95` to `0.80` with a small `ENSEMBLE_M = 4`, will the optimal spread shift **left** (under-dispersed) or **right** (over-dispersed)?
> - **Run:** Execute the next cell with the default sliders (`ENSEMBLE_M = 8`, `CRPS_ALPHA = 0.95`), then re-run with `ENSEMBLE_M = 4` and `CRPS_ALPHA = 0.80` to test your second prediction.
> - **What to observe:** Notice that both curves form clean U-shaped bowls bottoming out right near `spread = 1.00` (proving they are proper scoring rules), and watch how lowering `CRPS_ALPHA` pulls the `afCRPS` minimum slightly left of `1.00`.

In [ ]:
# @title 2. Explore afCRPS & Energy Score vs. Ensemble Spread
ENSEMBLE_M = 8 # @param {type:"slider", min:2, max:32, step:2}
CRPS_ALPHA = 0.95 # @param {type:"slider", min:0.80, max:1.00, step:0.01}

torch.manual_seed(0)
B, D = 512, 16
y_true = torch.randn(B, D)  # unit-variance true weather outcomes
base_noise = torch.randn(B, ENSEMBLE_M, D)

spread_grid = np.linspace(0.25, 2.25, 41)
afcrps_vals, es_vals = [], []
for s in spread_grid:
    samp = s * base_noise
    afcrps_vals.append(float(afcrps(samp, y_true, alpha=CRPS_ALPHA)))
    es_vals.append(float(energy_score(samp, y_true)) / math.sqrt(D))

best_s_crps = spread_grid[int(np.argmin(afcrps_vals))]
best_s_es = spread_grid[int(np.argmin(es_vals))]
print(f"Optimal spread minimizing afCRPS (α={CRPS_ALPHA:.2f}, m={ENSEMBLE_M}): {best_s_crps:.2f} (true spread = 1.00)")
print(f"Optimal spread minimizing Energy Score / √D:                      {best_s_es:.2f} (true spread = 1.00)")

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(spread_grid, afcrps_vals, lw=2.2, color="#2563eb", label=f"afCRPS (α={CRPS_ALPHA:.2f})")
ax.plot(spread_grid, es_vals, lw=2.2, color="#ea580c", label="Energy Score / √D")
ax.axvline(1.0, ls="--", color="k", alpha=0.6, label="True spread = 1.0")
ax.axvline(best_s_crps, ls=":", color="#2563eb", lw=1.6)
ax.set(title="Proper Scoring Rules Peak at the True Ensemble Spread", xlabel="Ensemble Spread Multiplier", ylabel="Loss (lower is better)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**How to interpret the plot:**
- Both `afCRPS` and `Energy Score` hit their minimum right around **spread = 1.0** (the true data spread), confirming that minimizing them drives a generator toward calibrated uncertainty.
- Lowering `CRPS_ALPHA` toward `0.80` with a small `ENSEMBLE_M` shifts the minimum slightly left (under-dispersed)—which is why Tempest uses `alpha = 0.95` to balance stability and calibration.

---

## 3. Fast Synthetic Training Demo: Baseline `afCRPS` vs. Physics-Aware Tail Losses

### Conceptual Intuition (Before the Math)
A rocket accelerating through max-Q doesn't care only about *average* wind speed—it cares intensely about **sudden vertical wind-shear sheets** (sharp changes in $(u, v)$ over a thin altitude layer) and **low-altitude vector wind** near the launch rail (`<= 300 m` AGL).

If you train a generator using only standard per-level `afCRPS`, the network plays it safe: it nails the typical profile, but **smooths away rare, violent shear sheets** because they occur only ~10–25% of the time and carry little weight in an unweighted average across 179 outputs.

Tempest fixes this in `tempest/generator/train.py:309-338` by adding two targeted physics terms:
1. **`shear_tail_loss` (Threshold-Weighted Tail CRPS):** Computes vector wind shear $S_l = \sqrt{\Delta u_l^2 + \Delta v_l^2} / \Delta z_{\text{km},l}$ at every layer, keeps only the **upper-tail excess** above the 90th percentile $q_{0.9,l}$ via $\mathrm{ReLU}(S_l - q_{0.9,l})$, and divides by $\max(\sigma_{\text{excess},l},\, 0.5)$ before applying `afcrps`.
   - *Why the `0.5` m/s/km floor on `shear_scale`?* Calm upper-stratospheric layers have tiny excess variance; without the `0.5` floor, dividing by a near-zero standard deviation would amplify noise into huge gradients.
2. **`low_wind_loss` (Rail-Exit Vector + Speed Loss):** Isolates all heights $\le 300\text{ m}$ AGL (`HEIGHTS <= 300.0`), scales winds by a characteristic boundary-layer speed of `3.0 m/s` (putting the term on the same $O(1)$ magnitude as standardized `afcrps`), and sums a **vector Energy Score** on $(u, v)$ (preserving wind *direction* + *speed* jointly) with a **speed `afcrps`** on $\sqrt{u^2 + v^2}$ (eliminating speed bias near the pad).

> **Predict → Run → Observe (Baseline vs. Physics-Aware Tail Shear)**
> - **Predict:** On a synthetic dataset where 25% of days contain a sharp low/mid-level wind-shear sheet: (1) Will the **Baseline (`afCRPS` only)** generator *over-predict* or *under-predict* the 90th-percentile peak vertical wind shear? (2) Will adding `shear_tail_loss` + `low_wind_loss` move the generated 90th-percentile shear closer to the synthetic truth?
> - **Run:** Execute the next cell (~15–25 seconds on CPU). Try setting `SHEAR_TAIL_WEIGHT = 0.0` vs. `1.5` afterward to isolate the tail-loss effect.
> - **What to observe:** Compare the printed `90th-percentile peak shear (m/s per km)` across **Truth**, **Baseline**, and **Physics-Aware**, and inspect the **right-hand histogram's upper tail** to see the blue Physics-Aware curve track the green Truth tail much more faithfully than the gray Baseline curve.

In [ ]:
# @title 3. Train Baseline vs. Physics-Aware Engression Generator (Fast CPU Demo)
TRAIN_STEPS = 70 # @param {type:"slider", min:30, max:120, step:10}
SHEAR_TAIL_WEIGHT = 1.0 # @param {type:"slider", min:0.0, max:2.0, step:0.25}
ES_WEIGHT = 0.1 # @param {type:"slider", min:0.0, max:0.5, step:0.05}
WIND_VEC_WEIGHT = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.25}

# Synthetic dataset of 1,024 profiles (u, v on the 44-height grid = 88 dims for speed)
# Includes realistic occasional strong shear sheets (heavy-tailed jumps around 1-3 km & jet level)
torch.manual_seed(7)
rng = np.random.default_rng(7)
N_DATA = 1024
n_h = len(HEIGHTS)
dz_km_np = np.diff(HEIGHTS) / 1000.0  # (43,)

cond_np = rng.normal(0, 1, size=(N_DATA, 6)).astype(np.float32)
# Build (N, 2, 44) physical (u, v) profiles with smooth jet + occasional sharp shear sheets
z_km = HEIGHTS / 1000.0
u_prof = np.zeros((N_DATA, n_h), dtype=np.float32)
v_prof = np.zeros((N_DATA, n_h), dtype=np.float32)
for i in range(N_DATA):
    base_u = 3.0 + 2.0 * cond_np[i, 0] + rng.normal(0, 1.5)
    base_v = 1.5 * cond_np[i, 1] + rng.normal(0, 1.5)
    jet = (14.0 + 5.0 * cond_np[i, 2] + rng.normal(0, 4.0)) * np.exp(-0.5 * ((z_km - 11.0) / 3.0) ** 2)
    # 20% of days contain a sharp low/mid-level wind-shear sheet
    sheet_u = np.zeros(n_h)
    sheet_v = np.zeros(n_h)
    if rng.random() < 0.25:
        zc = rng.uniform(0.5, 4.0)
        amp = rng.exponential(6.5)
        ang = rng.uniform(0, 2 * np.pi)
        step = 0.5 * (1.0 + np.tanh((z_km - zc) / 0.18))
        sheet_u = amp * np.cos(ang) * step
        sheet_v = amp * np.sin(ang) * step
    u_prof[i] = base_u + jet + sheet_u + rng.normal(0, 0.4, n_h)
    v_prof[i] = base_v + 0.3 * jet + sheet_v + rng.normal(0, 0.4, n_h)

# Encode into vertical-difference targets (N, 88) and standardize
raw_uv = np.stack([to_diff(u_prof), to_diff(v_prof)], axis=1).reshape(N_DATA, 2 * n_h)
tm = raw_uv.mean(axis=0, keepdims=True)
ts = raw_uv.std(axis=0, keepdims=True) + 1e-6
y_norm = torch.tensor((raw_uv - tm) / ts, dtype=torch.float32)
c_tens = torch.tensor(cond_np, dtype=torch.float32)
tm_t = torch.tensor(tm[0], dtype=torch.float32)
ts_t = torch.tensor(ts[0], dtype=torch.float32)
dz_km_t = torch.tensor(dz_km_np, dtype=torch.float32)

# Precompute 90th-percentile shear threshold & excess scale (matching train.py)
du_all = raw_uv[:, 1:n_h]
dv_all = raw_uv[:, n_h + 1:]
sh_all = np.hypot(du_all, dv_all) / dz_km_np[None, :]
shear_thr = torch.tensor(np.quantile(sh_all, 0.90, axis=0), dtype=torch.float32)
excess = np.maximum(sh_all - shear_thr.numpy()[None, :], 0.0)
shear_scale = torch.tensor(np.maximum(excess.std(axis=0), 0.5), dtype=torch.float32)
low_idx = torch.tensor(np.flatnonzero(HEIGHTS <= 300.0), dtype=torch.long)

def to_phys_uv(x_norm):
    d = (x_norm * ts_t + tm_t).reshape(*x_norm.shape[:-1], 2, n_h)
    return torch.cumsum(d, dim=-1)

# Exact formulas from tempest/generator/train.py:309-338
def shear_tail_loss(samples, target):
    def f(x):
        d = (x * ts_t + tm_t).reshape(*x.shape[:-1], 2, n_h)[..., 1:]
        sh = torch.hypot(d[..., 0, :], d[..., 1, :]) / dz_km_t
        return torch.relu(sh - shear_thr) / shear_scale
    return afcrps(f(samples), f(target), alpha=0.95)

def low_wind_loss(samples, target):
    ps, pt = to_phys_uv(samples), to_phys_uv(target)
    us, vs = ps[..., 0, low_idx], ps[..., 1, low_idx]
    ut, vt = pt[..., 0, low_idx], pt[..., 1, low_idx]
    es = energy_score(torch.stack([us, vs], -1).flatten(-2) / 3.0, torch.stack([ut, vt], -1).flatten(-2) / 3.0) / math.sqrt(2 * len(low_idx))
    sp = afcrps(torch.hypot(us, vs) / 3.0, torch.hypot(ut, vt) / 3.0, alpha=0.95)
    return es + sp

def train_quick(use_physics_losses: bool):
    torch.manual_seed(123)
    model = Generator(n_cond=6, n_out=2 * n_h, noise_dim=16, width=96, depth=2)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    hist = []
    for step in range(TRAIN_STEPS):
        idx = torch.randint(0, N_DATA, (128,))
        cb, yb = c_tens[idx], y_norm[idx]
        s = model(cb, m=4)
        loss = afcrps(s, yb, alpha=0.95)
        if use_physics_losses:
            loss = (
                loss
                + ES_WEIGHT * energy_score(s, yb) / math.sqrt(yb.shape[-1])
                + SHEAR_TAIL_WEIGHT * shear_tail_loss(s, yb)
                + WIND_VEC_WEIGHT * low_wind_loss(s, yb)
            )
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(float(loss.detach()))
    return model, hist

model_base, hist_base = train_quick(use_physics_losses=False)
model_phys, hist_phys = train_quick(use_physics_losses=True)

# Evaluate max vertical wind shear distribution on 256 test conditions
with torch.no_grad():
    s_base = model_base(c_tens[:256], m=8)
    s_phys = model_phys(c_tens[:256], m=8)

def max_shear_distribution(x_norm):
    d = (x_norm * ts_t + tm_t).reshape(*x_norm.shape[:-1], 2, n_h)[..., 1:]
    sh = torch.hypot(d[..., 0, :], d[..., 1, :]) / dz_km_t
    return sh.max(dim=-1).values.reshape(-1).numpy()

sh_true = max_shear_distribution(y_norm[:256])
sh_base = max_shear_distribution(s_base)
sh_phys = max_shear_distribution(s_phys)

print(f"90th-percentile peak shear (m/s per km) | Truth: {np.quantile(sh_true, 0.9):.2f} | Baseline: {np.quantile(sh_base, 0.9):.2f} | Physics-Aware: {np.quantile(sh_phys, 0.9):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].plot(hist_base, lw=2, color="#64748b", label="Baseline (afCRPS only)")
axes[0].plot(hist_phys, lw=2, color="#2563eb", label="Physics-Aware (+ES +ShearTail +LowWind)")
axes[0].set(title="Training Loss Trajectory", xlabel="Gradient Step", ylabel="Composite Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

bins = np.linspace(0, np.quantile(sh_true, 0.99) * 1.15, 30)
axes[1].hist(sh_true, bins=bins, density=True, alpha=0.35, color="#16a34a", label="Synthetic Truth")
axes[1].hist(sh_base, bins=bins, density=True, histtype="step", lw=2.2, color="#64748b", label="Baseline (afCRPS only)")
axes[1].hist(sh_phys, bins=bins, density=True, histtype="step", lw=2.2, color="#2563eb", label="Physics-Aware")
axes[1].set(title="Peak Vertical Wind Shear Distribution", xlabel="Max Layer Shear (m/s per km)", ylabel="Density")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Left:** Both models converge smoothly on CPU in under 30 seconds (the Physics-Aware curve starts higher simply because it sums four loss terms instead of one).
- **Right:** Focus on the upper-right tail of the peak vertical wind-shear histogram—adding `shear_tail_loss` pulls the generated ensemble's 90th-percentile shear closer to the true heavy-tailed distribution instead of washing out rocket-critical shear sheets.